# Python for Undergraduate Physics Lab Reports

This guided workbook shows how to turn measurements into clear, reproducible results with Python. The worked example uses a simulated optical spectrum, so it runs without external files or lab equipment. Later you can replace the simulated data with measurements from your own experiment.

**How to use this notebook:** run cells from top to bottom; change values and rerun the affected cells; keep units visible; and record every processing choice that could change a result. The final section contains practice tasks to try on your own.

## Learning goals and route through the notebook

By the end, you should be able to:

1. Write and call functions, and explain parameters, return values, local variables, and docstrings.
2. Use NumPy arrays for measured values and basic uncertainty calculations.
3. Read, inspect, clean, and summarize tabular measurements with Pandas.
4. Make labelled Matplotlib figures suitable for a lab report.
5. Detect and interpret peaks with SciPy's signal-processing tools.

The route is: function fundamentals → NumPy and uncertainty → Pandas data handling → Matplotlib figures → SciPy peak detection → report-ready results and practice.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, peak_widths
from io import StringIO

# Use a fixed seed so the simulated measurements are reproducible.
rng = np.random.default_rng(42)

## 1. Functions: reusable pieces of analysis

A function packages a calculation under a name. The values in parentheses are **parameters**; a call supplies **arguments**. **return** sends a result back to the caller. Variables created inside a function are local to that call, so return values rather than relying on hidden global state. A docstring between triple quotes describes the purpose, inputs, units, and output.

The examples below include a default argument, an explicit return value, input checks, and a calculation of uncertainty. Good functions do one clearly described task.

In [ ]:
def celsius_to_kelvin(temperature_c):
    """Convert a temperature in degrees Celsius to kelvin."""
    return temperature_c + 273.15


def mean_and_standard_error(values):
    """Return the sample mean and standard error for 1D repeated measurements."""
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or values.size < 2:
        raise ValueError("Provide at least two one-dimensional measurements.")
    mean = np.mean(values)
    standard_error = np.std(values, ddof=1) / np.sqrt(values.size)
    return mean, standard_error


def frequency_from_period(period_s, period_uncertainty_s):
    """Propagate period uncertainty to frequency for frequency = 1 / period."""
    if period_s <= 0 or period_uncertainty_s < 0:
        raise ValueError("Period must be positive and uncertainty non-negative.")
    frequency_hz = 1 / period_s
    frequency_uncertainty_hz = period_uncertainty_s / period_s**2
    return frequency_hz, frequency_uncertainty_hz


def gaussian(x, amplitude, centre, sigma, offset=0.0):
    """Evaluate a Gaussian peak; x, centre, and sigma must use the same units."""
    if sigma <= 0:
        raise ValueError("sigma must be positive.")
    return offset + amplitude * np.exp(-0.5 * ((x - centre) / sigma) ** 2)


# Positional and keyword arguments both supply the function's parameters.
room_temperature_k = celsius_to_kelvin(20.0)
period_mean_s, period_sem_s = mean_and_standard_error([0.502, 0.498, 0.501, 0.499])
frequency_hz, frequency_uncertainty_hz = frequency_from_period(period_mean_s, period_sem_s)
print(f"Room temperature: {room_temperature_k:.2f} K")
print(f"Mean period: {period_mean_s:.4f} ± {period_sem_s:.4f} s (standard error)")
print(f"Frequency: {frequency_hz:.3f} ± {frequency_uncertainty_hz:.3f} Hz")
print(f"Gaussian value at its centre: {gaussian(5.0, amplitude=2.0, centre=5.0, sigma=1.0):.2f}")

### Function practice

Before moving on, predict the output of the last call, then change its centre and sigma. Try calling a function with named arguments in a different order. Notice that the name **values** inside the mean function is local: it does not create a new notebook variable called **values**.

When writing your own analysis function, state the units and expected input shape. Avoid printing from a calculation function when you need the result later; return it, then decide separately how to display or save it.

## 2. NumPy arrays and measurement uncertainty

NumPy arrays store numerical measurements efficiently and support element-by-element operations. Keep related arrays the same length and in the same row order. Use variable names and comments that retain units; NumPy itself does not know whether a number represents seconds, metres, or volts.

In [ ]:
# A Python list and a NumPy array behave differently under arithmetic.
counts_list = [10, 12, 15]
counts = np.array(counts_list, dtype=float)
print("Counts doubled:", counts * 2)

# Element-wise division: each distance is divided by the corresponding time.
distance_m = np.array([0.50, 0.75, 1.00])
time_s = np.array([1.42, 2.00, 2.81])
velocity_m_per_s = distance_m / time_s
print("Velocity (m/s):", velocity_m_per_s)
print("Array shape:", velocity_m_per_s.shape)

### Repeated measurements and propagated uncertainty

For repeated measurements, the sample standard deviation describes their spread. The standard error of the mean (SEM) estimates the uncertainty in the mean under the usual independent, repeatable-measurement assumptions. These are not interchangeable. For a derived quantity, propagate uncertainty using the measurement model and assumptions appropriate to the experiment.

For frequency calculated as f = 1/T, first-order propagation gives σ_f = σ_T/T². The function above implements that relation. This example does not include calibration, resolution, or systematic uncertainties.

In [ ]:
period_trials_s = np.array([0.502, 0.498, 0.501, 0.499, 0.503])
period_mean_s, period_sem_s = mean_and_standard_error(period_trials_s)
frequency_hz, frequency_sem_hz = frequency_from_period(period_mean_s, period_sem_s)
print(f"n = {period_trials_s.size} trials")
print(f"Period = {period_mean_s:.4f} ± {period_sem_s:.4f} s (mean ± SEM)")
print(f"Frequency = {frequency_hz:.3f} ± {frequency_sem_hz:.3f} Hz (propagated SEM only)")

## 3. Pandas: load, inspect, and prepare tabular data

A Pandas DataFrame is a labelled table: columns have names, and each row can represent one observation. Here we create a reproducible simulated spectrum with wavelength, measured intensity, and a known illustrative background. A StringIO buffer lets read_csv parse CSV text without needing an external file.

For your own data, the usual starting point is **pd.read_csv("your_measurements.csv")**. Check the delimiter, header row, units, and missing values rather than assuming the file has the structure you expect.

In [ ]:
wavelength_nm = np.linspace(400.0, 700.0, 3001)
background_au = 0.05 + 0.0001 * (wavelength_nm - 400.0)
true_signal_au = (
    gaussian(wavelength_nm, 0.85, 486.1, 1.5)
    + gaussian(wavelength_nm, 0.65, 589.0, 2.0)
    + gaussian(wavelength_nm, 0.95, 656.3, 1.2)
)
measurement_noise_au = rng.normal(0.0, 0.02, wavelength_nm.size)
intensity_au = background_au + true_signal_au + measurement_noise_au

spectrum = pd.DataFrame({
    "wavelength_nm": wavelength_nm,
    "intensity_au": intensity_au,
    "background_au": background_au,
    "uncertainty_au": np.full(wavelength_nm.size, 0.02),
})

csv_buffer = StringIO()
spectrum.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)
spectrum = pd.read_csv(csv_buffer)
print(f"Loaded {len(spectrum)} simulated rows from CSV text.")

### Inspect before analysing

Start with shape, column names, data types, a few rows, and summary statistics. Decide how to treat missing or invalid values based on what they mean physically. Silently dropping data can bias results; this example only demonstrates how to make an explicit cleaned copy.

In [ ]:
print("Shape:", spectrum.shape)
print("Columns:", spectrum.columns.tolist())
print("Data types:\n", spectrum.dtypes)
display(spectrum.head())
print("Missing values per column:\n", spectrum.isna().sum())
display(spectrum.describe())

clean_spectrum = (
    spectrum.dropna(subset=["wavelength_nm", "intensity_au"])
    .sort_values("wavelength_nm")
    .reset_index(drop=True)
    .copy()
)
clean_spectrum["excess_au"] = clean_spectrum["intensity_au"] - clean_spectrum["background_au"]

## 4. Matplotlib: figures that communicate measurements

A lab-report plot should identify the measured quantity and units on both axes, show the data and uncertainty honestly, and use a concise title or caption. Keep text legible and avoid decorative styling that distracts from the measurement. Use error bars when uncertainties are known and make clear what they represent.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
step = 100  # show error bars on a subset so they remain readable
ax.errorbar(
    clean_spectrum["wavelength_nm"][::step],
    clean_spectrum["intensity_au"][::step],
    yerr=clean_spectrum["uncertainty_au"][::step],
    fmt="o",
    markersize=3,
    capsize=2,
    alpha=0.7,
    label="sampled measurements ± 1σ",
)
ax.plot(
    clean_spectrum["wavelength_nm"],
    clean_spectrum["intensity_au"],
    linewidth=1,
    alpha=0.65,
    label="measured spectrum",
)
ax.set(xlabel="Wavelength (nm)", ylabel="Intensity (arbitrary units)", title="Simulated spectrum")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

# When ready to include the figure in a report, save it with a descriptive filename:
# fig.savefig("spectrum_overview.png", dpi=300, bbox_inches="tight")

## 5. SciPy: detect peaks and inspect their properties

SciPy's **find_peaks** identifies local maxima in a one-dimensional signal. Its settings have physical consequences:

- **prominence** measures how far a peak rises above its surrounding contour; it can reject small noise bumps.
- **distance** is a minimum separation in **samples**, not wavelength units.
- **width** is also reported in samples. Convert to wavelength units using the sampling interval.

Thresholds are analysis choices, not facts about the instrument. Inspect the signal and test whether the detected peaks change when reasonable thresholds are varied. Here we subtract the known simulated background, use evenly spaced wavelengths, and set a 3 nm minimum separation.

In [ ]:
signal = clean_spectrum["excess_au"].to_numpy()
wavelength = clean_spectrum["wavelength_nm"].to_numpy()
sample_spacing_nm = np.median(np.diff(wavelength))
minimum_separation_nm = 3.0
minimum_distance_samples = int(np.ceil(minimum_separation_nm / sample_spacing_nm))

peak_indices, properties = find_peaks(
    signal,
    prominence=0.10,
    distance=minimum_distance_samples,
    width=5,
)
widths_samples, width_heights, left_ips, right_ips = peak_widths(
    signal, peak_indices, rel_height=0.5
)

peak_results = pd.DataFrame({
    "wavelength_nm": wavelength[peak_indices],
    "intensity_au": clean_spectrum["intensity_au"].to_numpy()[peak_indices],
    "prominence_au": properties["prominences"],
    "width_nm": widths_samples * sample_spacing_nm,
})
print(f"Detected {len(peak_results)} peaks; sample spacing = {sample_spacing_nm:.3f} nm")
display(peak_results.round(3))

### Check and visualise the detections

The peak location above is the wavelength of a sampled data point, not a fitted centre. The width at half prominence is a shape descriptor; it is not automatically a measurement uncertainty or a spectral resolution. For irregularly spaced measurements, do not convert sample widths with one constant spacing—resample carefully or use a model in the measured coordinates.

Always overlay detections on the original data. This makes missed peaks, noise detections, and unsuitable thresholds easier to notice.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), layout="constrained")
ax.plot(wavelength, clean_spectrum["intensity_au"], linewidth=1, label="measured spectrum")
ax.scatter(
    peak_results["wavelength_nm"],
    peak_results["intensity_au"],
    color="crimson",
    zorder=3,
    label="detected peaks",
)
for row in peak_results.itertuples():
    ax.annotate(f"{row.wavelength_nm:.1f} nm", (row.wavelength_nm, row.intensity_au),
                xytext=(0, 8), textcoords="offset points", ha="center", fontsize=8)
ax.set(xlabel="Wavelength (nm)", ylabel="Intensity (arbitrary units)", title="Peak detection check")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

# Optional report output:
# fig.savefig("spectrum_with_peaks.png", dpi=300, bbox_inches="tight")

## 6. Presenting results responsibly

A useful result includes the measurement and unit, how it was obtained, and enough information to reproduce the analysis. For this example, report the peak-detection thresholds and the wavelength sampling interval. Do **not** report the sampling interval as the uncertainty: calibration, resolution, noise, and the analysis model may all contribute.

The formatted output below is a starting point for a results table. Choose significant figures based on the measurement uncertainty and the conventions of your lab.

In [ ]:
print("Peak results (sampled peak locations; widths at half prominence):")
for row in peak_results.itertuples():
    print(
        f"λ = {row.wavelength_nm:.1f} nm, "
        f"intensity = {row.intensity_au:.3f} a.u., "
        f"prominence = {row.prominence_au:.3f} a.u., "
        f"width = {row.width_nm:.2f} nm"
    )

# To create a CSV for a report or spreadsheet, uncomment and choose an output path:
# peak_results.to_csv("detected_peaks.csv", index=False)

## 7. Practice exercises

Try these after running the worked example. Write your solution in new cells below each prompt, and explain what you expect before running the code.

1. **Function:** write a function that calculates the period of a simple pendulum, T = 2π√(L/g). Give the inputs and output clear names with units, and test at least two lengths.
2. **Uncertainty:** collect five simulated pendulum periods. Use the mean and SEM, then propagate uncertainty to frequency. State which sources of uncertainty your calculation leaves out.
3. **Pandas:** alter the spectrum example to include a second trial column. Use a DataFrame to calculate the mean intensity by wavelength and inspect the result.
4. **Plotting:** choose a smaller wavelength region around one peak. Plot it with labelled axes, units, a legend, and readable uncertainty bars. Save a report-quality figure.
5. **Peak detection:** vary prominence and minimum separation. How do the number, locations, and widths of reported peaks change? Add a short note justifying your final choices.
6. **Real data:** load one of your own CSV files with pd.read_csv. Inspect its columns and units before plotting or processing it. Adapt the code rather than assuming its column names match this example.

**Hints:** Python's square root and π are available from NumPy. For a smaller wavelength region, use a Boolean mask on the wavelength array and apply the same mask to the intensity array. Keep the original data intact while testing thresholds.

## Lab-report analysis checklist

- Can another person tell what each array and DataFrame column represents, including units?
- Are functions documented, tested on simple cases, and returning values rather than hiding results in global variables?
- Have you inspected the raw data and justified any excluded points or cleaning steps?
- Does each figure have readable labels, units, a legend where needed, and honest uncertainty bars?
- Have you stated the peak-detection method and thresholds, and checked detections visually?
- Are uncertainty and significant figures justified by the measurement model rather than the number of digits printed by Python?

A computational result supports a physical interpretation; it does not replace checking assumptions, uncertainty, or the experiment itself.